<a href="https://colab.research.google.com/github/deepan98raj-dotcom/Capstone_project/blob/main/phase1_to_5_resume_parser_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Phase 1 to 5 Resume Parser System

 ```    

       [ PDF Upload ]
             │
             ▼
┌───────────────────────────┐
│ Phase 1: PyMuPDF Parser   │ ──► Extracts Text & Missing Data
└────────────┬──────────────┘
             │
             ▼
┌───────────────────────────┐
│ Phase 2: Relational DB    │ ──► PostgreSQL / SQLite
│          & Qdrant Vector  │ ──► Generates 384-dim Embeddings
└────────────┬──────────────┘
             │
             ▼
┌───────────────────────────┐
│ Phase 3: Dynamic RAG      │ ──► Generates Gap-Filling &
│          Q&A Engine       │     Skill Validation Questions
└────────────┬──────────────┘
             │
             ▼
┌───────────────────────────┐
│ Phase 4: Evaluation Agent │ ──► 1 to 10 Scoring Matrix
│          & Ranking Engine │ ──► Weighted Composite Semantic Rank
└───────────────────────────┘

```




#1.Resume Parsing & Missing Data Audit

In [ ]:
!pip install fastapi uvicorn pymupdf pydantic pyngrok requests

In [ ]:
%%writefile main.py
import re
import fitz  # PyMuPDF
from typing import List, Dict, Optional
from pydantic import BaseModel, Field
from fastapi import FastAPI, UploadFile, File, HTTPException

app = FastAPI(
    title="Resume Parsing API - Phase One",
    description="Parses PDF resumes into structured JSON and detects missing essential data.",
    version="1.0.0"
)

# ------------------------------------------------------------------
# Data Models
# ------------------------------------------------------------------

class EducationItem(BaseModel):
    degree: Optional[str] = None
    institution: Optional[str] = None
    year: Optional[str] = None

class WorkItem(BaseModel):
    title: Optional[str] = None
    company: Optional[str] = None
    duration: Optional[str] = None
    description: Optional[str] = None

class ResumeData(BaseModel):
    education: List[EducationItem] = Field(default_factory=list)
    work_experience: List[WorkItem] = Field(default_factory=list)
    core_skills: List[str] = Field(default_factory=list)

class MissingDataReport(BaseModel):
    has_missing_data: bool
    missing_education_fields: List[str] = Field(default_factory=list)
    missing_skills: List[str] = Field(default_factory=list)
    general_warnings: List[str] = Field(default_factory=list)

class ResumeParseResponse(BaseModel):
    filename: str
    parsed_data: ResumeData
    missing_data_report: MissingDataReport

# ------------------------------------------------------------------
# Text Extraction & Parsing Engine
# ------------------------------------------------------------------

SKILLS_TAXONOMY = [
    "Generative AI", "Machine Learning", "Deep Learning", "NLP", "LLM", "RAG", "MCP",
    "Prompt Engineering", "Image Processing", "Model Tuning", "PyTorch", "TensorFlow",
    "Python", "C++", "Embedded C", "Git", "Version Control", "SQL", "FastAPI", "Docker",
    "Auto CAD", "Optitex", "ProCAM", "AMS Programming", "Juki PM-1", "Brother PS-300B",
    "Embedded System CAD", "Adobe Photoshop", "Adobe Illustrator", "MS Excel", "MS Office"
]

def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    try:
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        full_text = [page.get_text("text") for page in doc if page.get_text("text")]
        doc.close()
        return "\n".join(full_text)
    except Exception as e:
        raise ValueError(f"Failed to extract text from PDF: {str(e)}")

def parse_resume_text(raw_text: str) -> ResumeData:
    lines = [line.strip() for line in raw_text.split("\n") if line.strip()]
    sections: Dict[str, List[str]] = {"summary": [], "skills": [], "work": [], "education": [], "other": []}
    current_section = "other"

    for line in lines:
        clean_lower = line.lower()
        if ("summary" in clean_lower or "profile" in clean_lower) and len(clean_lower) < 35:
            current_section = "summary"
            continue
        elif ("skills" in clean_lower or "competencies" in clean_lower) and len(clean_lower) < 35:
            current_section = "skills"
            continue
        elif ("experience" in clean_lower or "employment" in clean_lower) and len(clean_lower) < 35:
            current_section = "work"
            continue
        elif ("education" in clean_lower or "academic" in clean_lower) and len(clean_lower) < 35:
            current_section = "education"
            continue
        sections[current_section].append(line)

    # Education Extraction
    education_list: List[EducationItem] = []
    edu_lines = sections["education"]
    i = 0
    while i < len(edu_lines):
        line = edu_lines[i]
        if any(deg in line for deg in ["Bachelor", "B.E.", "Diploma", "Post Graduate", "PDFT", "Master"]):
            degree = line
            institution, year = None, None
            if i + 1 < len(edu_lines):
                next_line = edu_lines[i + 1]
                years = re.findall(r"\b(19\d{2}|20\d{2})\b", next_line)
                if years:
                    year = " - ".join(years) if len(years) > 1 else years[0]
                institution = next_line
                i += 1
            education_list.append(EducationItem(degree=degree, institution=institution, year=year))
        i += 1

    # Work Experience Extraction
    work_list: List[WorkItem] = []
    work_lines = sections["work"]
    current_work, desc_buffer = None, []

    for line in work_lines:
        date_match = re.search(r"\((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)?\s*\d{4}\s*[–-]\s*(?:Present|(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)?\s*\d{4})\)", line, re.IGNORECASE)
        if date_match or ("|" in line and any(yr in line for yr in ["2020", "2021", "2022", "2023", "2024", "2025", "2026"])):
            if current_work:
                current_work.description = " ".join(desc_buffer)
                work_list.append(current_work)
                desc_buffer = []
            parts = line.split("|") if "|" in line else [line]
            company = parts[0].strip() if len(parts) > 0 else line
            title = parts[1].strip() if len(parts) > 1 else company
            duration = date_match.group(0) if date_match else None
            current_work = WorkItem(title=title, company=company, duration=duration)
        elif current_work:
            desc_buffer.append(line)

    if current_work:
        current_work.description = " ".join(desc_buffer)
        work_list.append(current_work)

    # Skills Extraction
    extracted_skills = set()
    for skill in SKILLS_TAXONOMY:
        if re.search(r"\b" + re.escape(skill) + r"\b", raw_text, re.IGNORECASE):
            extracted_skills.add(skill)

    return ResumeData(
        education=education_list,
        work_experience=work_list,
        core_skills=sorted(list(extracted_skills))
    )

def analyze_missing_data(data: ResumeData) -> MissingDataReport:
    missing_edu, missing_skills, warnings = [], [], []

    if not data.education:
        missing_edu.append("No Education section detected.")
    else:
        for idx, edu in enumerate(data.education, start=1):
            if not edu.degree: missing_edu.append(f"Education #{idx}: Missing degree.")
            if not edu.institution: missing_edu.append(f"Education #{idx}: Missing institution.")
            if not edu.year: missing_edu.append(f"Education #{idx}: Missing year.")

    if not data.core_skills:
        missing_skills.append("No core skills extracted.")
    elif len(data.core_skills) < 5:
        warnings.append(f"Only {len(data.core_skills)} skills found. Consider adding more.")

    return MissingDataReport(
        has_missing_data=bool(missing_edu or missing_skills),
        missing_education_fields=missing_edu,
        missing_skills=missing_skills,
        general_warnings=warnings
    )

# ------------------------------------------------------------------
# Endpoint
# ------------------------------------------------------------------

@app.post("/parse-resume", response_model=ResumeParseResponse)
async def parse_resume_file(file: UploadFile = File(...)):
    if file.content_type != "application/pdf":
        raise HTTPException(status_code=400, detail="Must be a PDF file.")

    pdf_bytes = await file.read()
    raw_text = extract_text_from_pdf_bytes(pdf_bytes)

    if not raw_text.strip():
        raise HTTPException(status_code=422, detail="Empty or unreadable PDF.")

    parsed_data = parse_resume_text(raw_text)
    missing_report = analyze_missing_data(parsed_data)

    return ResumeParseResponse(
        filename=file.filename or "resume.pdf",
        parsed_data=parsed_data,
        missing_data_report=missing_report
    )

Overwriting main.py


In [ ]:
import fitz
import json
from fastapi.testclient import TestClient
from main import app

# 1. Initialize Client
client = TestClient(app)

# 2. Sample Resume Text
sample_resume_text = """
DEEPAN RAJ K
Chennai, Tamil Nadu, India | +91 8939215805 | deepan98raj@gmail.com

PROFESSIONAL SUMMARY
Electronics and Communication Engineer with experience in CAD design automation, embedded systems, and AI-assisted tool workflows. Expertise in Python, machine learning, Deep learning, NLP, LLM, RAG, and MCP.

CORE COMPETENCIES & TECHNICAL SKILLS
• AI & Machine Learning: Generative AI, Prompt Engineering, Image Processing
• Programming: Python, Embedded C, SQL
• CAD & Automation: Auto CAD, Optitex 2D, ProCAM, AMS Programming

PROFESSIONAL EXPERIENCE
BHARTIYA INTERNATIONAL LTD | Pattern / AI Engineer (Jul 2024 – AUG 2026)
• Leveraged generative AI tools for visual algorithms and art design.

EDUCATION
Bachelor of Engineering (B.E.) in Electronics and Communication Engineering
SRM Easwari Engineering College | 2015 – 2019
"""

# 3. Build sample PDF in memory
doc = fitz.open()
page = doc.new_page()
page.insert_text((40, 40), sample_resume_text, fontsize=9)
pdf_bytes = doc.write()
doc.close()

# 4. Send POST request
response = client.post(
    "/parse-resume",
    files={"file": ("Deepan_Raj_K_Resume.pdf", pdf_bytes, "application/pdf")},
)

# 5. Output Result
print("Status Code:", response.status_code)
print(json.dumps(response.json(), indent=2))

Status Code: 200
{
  "filename": "Deepan_Raj_K_Resume.pdf",
  "parsed_data": {
    "education": [
      {
        "degree": "Bachelor of Engineering (B.E.) in Electronics and Communication Engineering",
        "institution": "SRM Easwari Engineering College | 2015 \u00b7 2019",
        "year": "2015 - 2019"
      }
    ],
    "work_experience": [
      {
        "title": "Pattern / AI Engineer (Jul 2024 \u00b7 AUG 2026)",
        "company": "BHARTIYA INTERNATIONAL LTD",
        "duration": null,
        "description": "\u00b7 Leveraged generative AI tools for visual algorithms and art design."
      }
    ],
    "core_skills": [
      "AMS Programming",
      "Auto CAD",
      "Embedded C",
      "Generative AI",
      "Image Processing",
      "Machine Learning",
      "Optitex",
      "ProCAM",
      "Prompt Engineering",
      "Python",
      "SQL"
    ]
  },
  "missing_data_report": {
    "has_missing_data": false,
    "missing_education_fields": [],
    "missing_skills": [],
    

# 2.Database (PostgreSQL) & Vector DB (Qdrant) Setup

In [ ]:
!pip install sqlalchemy psycopg2-binary qdrant-client sentence-transformers pydantic

In [ ]:
%%writefile database.py
import os
from datetime import datetime
from typing import List, Dict, Any
from pydantic import BaseModel
from sqlalchemy import create_engine, Column, Integer, String, Text, Float, DateTime, ForeignKey, JSON
from sqlalchemy.orm import declarative_base, sessionmaker, relationship
from qdrant_client import QdrantClient
from qdrant_client.http import models as qdrant_models
from sentence_transformers import SentenceTransformer

DATABASE_URL = os.getenv("DATABASE_URL", "sqlite:///./phase2_database.db")

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False} if "sqlite" in DATABASE_URL else {}
)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

class CandidateProfile(Base):
    __tablename__ = "candidates"

    id = Column(Integer, primary_key=True, index=True)
    full_name = Column(String(150), nullable=False)
    email = Column(String(150), unique=True, index=True, nullable=False)
    phone = Column(String(50), nullable=True)
    location = Column(String(100), nullable=True)
    resume_json = Column(JSON, nullable=False)
    missing_data_report = Column(JSON, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

    assessments = relationship("Assessment", back_populates="candidate", cascade="all, delete-orphan")

class Assessment(Base):
    __tablename__ = "assessments"

    id = Column(Integer, primary_key=True, index=True)
    candidate_id = Column(Integer, ForeignKey("candidates.id"), nullable=False)
    target_role = Column(String(100), nullable=False)
    overall_score = Column(Float, nullable=True)
    status = Column(String(50), default="PENDING")
    summary_feedback = Column(Text, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

    candidate = relationship("CandidateProfile", back_populates="assessments")
    evaluations = relationship("Evaluation", back_populates="assessment", cascade="all, delete-orphan")

class Evaluation(Base):
    __tablename__ = "evaluations"

    id = Column(Integer, primary_key=True, index=True)
    assessment_id = Column(Integer, ForeignKey("assessments.id"), nullable=False)
    category = Column(String(50), nullable=False)
    question = Column(Text, nullable=False)
    candidate_answer = Column(Text, nullable=True)
    score = Column(Float, nullable=True)
    logical_explanation = Column(Text, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

    assessment = relationship("Assessment", back_populates="evaluations")

def init_db():
    Base.metadata.create_all(bind=engine)

class VectorDBManager:
    def __init__(self, collection_name: str = "candidates_index"):
        self.collection_name = collection_name
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.vector_size = self.encoder.get_sentence_embedding_dimension()
        self.client = QdrantClient(":memory:")
        self._setup_collection()

    def _setup_collection(self):
        collections = self.client.get_collections().collections
        exists = any(c.name == self.collection_name for c in collections)
        if not exists:
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=qdrant_models.VectorParams(
                    size=self.vector_size,
                    distance=qdrant_models.Distance.COSINE
                )
            )

    def upsert_candidate_vector(self, candidate_id: int, full_name: str, skills: List[str], work_summary: str):
        text_repr = f"Candidate: {full_name}. Skills: {', '.join(skills)}. Experience: {work_summary}"
        vector = self.encoder.encode(text_repr).tolist()

        self.client.upsert(
            collection_name=self.collection_name,
            points=[
                qdrant_models.PointStruct(
                    id=candidate_id,
                    vector=vector,
                    payload={
                        "candidate_id": candidate_id,
                        "full_name": full_name,
                        "skills": skills,
                        "summary": work_summary,
                    }
                )
            ]
        )

    def search_candidates_by_role(self, role_requirements: str, limit: int = 5) -> List[Dict[str, Any]]:
        query_vector = self.encoder.encode(role_requirements).tolist()
        search_results = self.client.search(
            collection_name=self.collection_name,
            query_vector=query_vector,
            limit=limit
        )

        results = []
        for point in search_results:
            results.append({
                "candidate_id": point.payload["candidate_id"],
                "full_name": point.payload["full_name"],
                "similarity_score": round(point.score, 4),
                "skills": point.payload["skills"],
            })
        return results

Writing database.py


In [ ]:
import json
import os
from datetime import datetime
from typing import Any, Dict, List
from qdrant_client import QdrantClient
from qdrant_client.http import models as qdrant_models
from sentence_transformers import SentenceTransformer
from sqlalchemy import (
    JSON,
    Column,
    DateTime,
    Float,
    ForeignKey,
    Integer,
    String,
    Text,
    create_engine,
)
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

# 1. Database Setup
DATABASE_URL = "sqlite:///./phase2_colab.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()


class CandidateProfile(Base):
    __tablename__ = "candidates"

    id = Column(Integer, primary_key=True, index=True)
    full_name = Column(String(150), nullable=False)
    email = Column(String(150), unique=True, index=True, nullable=False)
    phone = Column(String(50), nullable=True)
    location = Column(String(100), nullable=True)
    resume_json = Column(JSON, nullable=False)
    missing_data_report = Column(JSON, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)


class VectorDBManager:

    def __init__(self, collection_name: str = "ai_candidates"):
        self.collection_name = collection_name
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.vector_size = self.encoder.get_sentence_embedding_dimension()
        self.client = QdrantClient(":memory:")
        self._setup_collection()

    def _setup_collection(self):
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=qdrant_models.VectorParams(
                size=self.vector_size, distance=qdrant_models.Distance.COSINE
            ),
        )

    def upsert_candidate_vector(
        self, candidate_id: int, full_name: str, skills: List[str], work_summary: str
    ):
        text_repr = f"Candidate: {full_name}. Skills: {', '.join(skills)}. Experience: {work_summary}"
        vector = self.encoder.encode(text_repr).tolist()
        self.client.upsert(
            collection_name=self.collection_name,
            points=[
                qdrant_models.PointStruct(
                    id=candidate_id,
                    vector=vector,
                    payload={
                        "candidate_id": candidate_id,
                        "full_name": full_name,
                        "skills": skills,
                        "summary": work_summary,
                    },
                )
            ],
        )

    def search_candidates_by_role(
        self, role_requirements: str, limit: int = 5
    ) -> List[Dict[str, Any]]:
        query_vector = self.encoder.encode(role_requirements).tolist()

        # Using query_points for qdrant-client >= 1.10.0 compatibility
        search_results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_vector,
            limit=limit,
        ).points

        return [
            {
                "candidate_id": point.payload["candidate_id"],
                "full_name": point.payload["full_name"],
                "similarity_score": round(point.score, 4),
                "skills": point.payload["skills"],
            }
            for point in search_results
        ]


# 2. Execution
Base.metadata.create_all(bind=engine)
db = SessionLocal()
vector_db = VectorDBManager()

sample_resume_json = {
    "education": [
        {
            "degree": "B.E. Electronics and Communication",
            "institution": "SRM Easwari Engineering College",
            "year": "2019",
        }
    ],
    "work_experience": [
        {
            "title": "Pattern / AI Engineer",
            "company": "BHARTIYA INTERNATIONAL LTD",
            "duration": "Jul 2024 - AUG 2026",
        }
    ],
    "core_skills": [
        "Generative AI",
        "Python",
        "RAG",
        "LLM",
        "Deep Learning",
        "Optitex 2D",
        "Auto CAD",
    ],
}

candidate = CandidateProfile(
    full_name="Deepan Raj K",
    email="deepan98raj@gmail.com",
    phone="+91 8939215805",
    location="Chennai, Tamil Nadu, India",
    resume_json=sample_resume_json,
    missing_data_report={"has_missing_data": False, "missing_fields": []},
)

# Clear table to avoid unique email constraint error on multiple runs
db.query(CandidateProfile).delete()
db.commit()

db.add(candidate)
db.commit()
db.refresh(candidate)

print(f"✓ DB Record Created. Candidate ID: {candidate.id}")

vector_db.upsert_candidate_vector(
    candidate_id=candidate.id,
    full_name=candidate.full_name,
    skills=sample_resume_json["core_skills"],
    work_summary="Pattern and AI Engineer experienced in Generative AI tools, prompt engineering, Python, and RAG architectures.",
)
print(f"✓ Candidate Vector Indexed in Qdrant Vector DB")

matches = vector_db.search_candidates_by_role(
    "Looking for an AI Engineer experienced in Generative AI, Python, RAG pipelines."
)

print("\n--- Semantic Similarity Ranking Results ---")
print(json.dumps(matches, indent=2))

db.close()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_4711/4001185405.py:46: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.vector_size = self.encoder.get_sentence_embedding_dimension()


✓ DB Record Created. Candidate ID: 1
✓ Candidate Vector Indexed in Qdrant Vector DB

--- Semantic Similarity Ranking Results ---
[
  {
    "candidate_id": 1,
    "full_name": "Deepan Raj K",
    "similarity_score": 0.7512,
    "skills": [
      "Generative AI",
      "Python",
      "RAG",
      "LLM",
      "Deep Learning",
      "Optitex 2D",
      "Auto CAD"
    ]
  }
]


# 3.Dynamic Q&A RAG Engine (Gap Filling & Skill Validation)

In [ ]:
import json
from typing import Dict, List, Optional
from pydantic import BaseModel, Field

# ------------------------------------------------------------------
# Phase 3 Data Models
# ------------------------------------------------------------------


class QuestionPrompt(BaseModel):
    category: (
        str  # GAP_FILLING, SKILL_VALIDATION, or PROBLEM_SOLVING_SCENARIO
    )
    question: str
    context_reference: str  # The specific skill/project/missing item triggering the question


class CandidateAnswer(BaseModel):
    question_id: int
    question: str
    answer: str


class ConversationState(BaseModel):
    candidate_id: int
    target_role: str
    chat_history: List[Dict[str, str]] = Field(default_factory=list)
    pending_questions: List[QuestionPrompt] = Field(default_factory=list)


# ------------------------------------------------------------------
# Phase 3 RAG Engine Logic
# ------------------------------------------------------------------


class DynamicRAGEngine:
    """Generates targeted candidate questions for gap-filling, project validation,"""

    # and practical scenario testing.

    def generate_assessment_questions(
        self, candidate_data: Dict
    ) -> List[QuestionPrompt]:
        questions: List[QuestionPrompt] = []

        resume_json = candidate_data.get("resume_json", {})
        missing_report = candidate_data.get("missing_data_report", {})

        # --- Rule 1: Gap Filling Questions ---
        if missing_report.get("has_missing_data"):
            for field in missing_report.get("missing_education_fields", []):
                questions.append(
                    QuestionPrompt(
                        category="GAP_FILLING",
                        question=f"We noticed some missing details: {field}. Could you please provide this information?",
                        context_reference="Education Gap",
                    )
                )

        # --- Rule 2: Targeted Skill & Project Validation ---
        skills = resume_json.get("core_skills", [])
        work_exp = resume_json.get("work_experience", [])

        # Validate AI / RAG skills if present
        if any(s in ["Generative AI", "RAG", "LLM"] for s in skills):
            questions.append(
                QuestionPrompt(
                    category="SKILL_VALIDATION",
                    question="You mentioned experience with Generative AI and RAG architectures. Can you describe how you handled chunking, embedding generation, or prompt engineering in your AI workflow?",
                    context_reference="Generative AI / RAG Skills",
                )
            )

        # Validate CAD / Automation / Control experience
        if any(
            s in ["AMS Programming", "Auto CAD", "Optitex 2D"] for s in skills
        ):
            questions.append(
                QuestionPrompt(
                    category="SKILL_VALIDATION",
                    question="In your pattern and AMS programming roles, how did you program automated sequence operations or build custom JIGs to optimize production?",
                    context_reference="CAD & Sequence Automation",
                )
            )

        # --- Rule 3: Problem Solving Scenario Questions ---
        if work_exp:
            latest_role = work_exp[0]
            company = latest_role.get("company", "your previous company")
            title = latest_role.get("title", "Engineer")

            questions.append(
                QuestionPrompt(
                    category="PROBLEM_SOLVING_SCENARIO",
                    question=f"During your work as a {title} at {company}, what was a major technical bottleneck or edge case you encountered, and what specific steps did you take to solve it?",
                    context_reference=f"Experience at {company}",
                )
            )

        return questions

    def add_to_chat_memory(
        self, state: ConversationState, role: str, message: str
    ):
        """Appends user or assistant messages to contextual conversation history."""
        state.chat_history.append({"role": role, "content": message})


# ------------------------------------------------------------------
# Test Phase 3 Workflow
# ------------------------------------------------------------------

rag_engine = DynamicRAGEngine()

# Candidate Profile Data (From Phase 1 & 2)
candidate_profile = {
    "candidate_id": 1,
    "resume_json": {
        "education": [
            {
                "degree": "B.E. Electronics and Communication",
                "institution": "SRM Easwari Engineering College",
                "year": "2019",
            }
        ],
        "work_experience": [
            {
                "title": "Pattern / AI Engineer",
                "company": "BHARTIYA INTERNATIONAL LTD",
                "duration": "Jul 2024 - AUG 2026",
            }
        ],
        "core_skills": [
            "Generative AI",
            "Python",
            "RAG",
            "LLM",
            "AMS Programming",
            "Optitex 2D",
        ],
    },
    "missing_data_report": {
        "has_missing_data": True,
        "missing_education_fields": [
            "Education #1: Missing exact percentage / CGPA record."
        ],
    },
}

# 1. Generate Questions
generated_questions = rag_engine.generate_assessment_questions(
    candidate_profile
)

# 2. Initialize Conversation Memory
state = ConversationState(
    candidate_id=1,
    target_role="AI Engineer",
    pending_questions=generated_questions,
)

# Output Generated Questions
print("--- Phase 3: Generated Dynamic RAG Questions ---")
for idx, q in enumerate(state.pending_questions, start=1):
    print(f"\nQuestion #{idx} [{q.category}]")
    print(f"Context: {q.context_reference}")
    print(f"Prompt:  {q.question}")

--- Phase 3: Generated Dynamic RAG Questions ---

Question #1 [GAP_FILLING]
Context: Education Gap
Prompt:  We noticed some missing details: Education #1: Missing exact percentage / CGPA record.. Could you please provide this information?

Question #2 [SKILL_VALIDATION]
Context: Generative AI / RAG Skills
Prompt:  You mentioned experience with Generative AI and RAG architectures. Can you describe how you handled chunking, embedding generation, or prompt engineering in your AI workflow?

Question #3 [SKILL_VALIDATION]
Context: CAD & Sequence Automation
Prompt:  In your pattern and AMS programming roles, how did you program automated sequence operations or build custom JIGs to optimize production?

Question #4 [PROBLEM_SOLVING_SCENARIO]
Context: Experience at BHARTIYA INTERNATIONAL LTD
Prompt:  During your work as a Pattern / AI Engineer at BHARTIYA INTERNATIONAL LTD, what was a major technical bottleneck or edge case you encountered, and what specific steps did you take to solve it?


# 4.Evaluation Scoring & Candidate Ranking Matrix

In [ ]:
import json
from typing import Dict, List, Optional
from pydantic import BaseModel, Field


# ------------------------------------------------------------------
# Phase 4 Data Models
# ------------------------------------------------------------------


class EvaluationCriteria(BaseModel):
    technical_accuracy: float = Field(
        ..., description="Score from 1.0 to 10.0"
    )
    completeness: float = Field(..., description="Score from 1.0 to 10.0")
    logical_rationale: float = Field(..., description="Score from 1.0 to 10.0")
    relevance: float = Field(..., description="Score from 1.0 to 10.0")


class QuestionGradingResult(BaseModel):
    question_id: int
    category: str
    overall_score: float = Field(
        ..., description="Weighted 1 to 10 score for this response"
    )
    criteria_breakdown: EvaluationCriteria
    logical_explanation: str


class CandidateRankResult(BaseModel):
    candidate_id: int
    full_name: str
    vector_similarity_score: float
    assessment_score: float
    final_weighted_rank_score: float
    ranking_summary: str


# ------------------------------------------------------------------
# Evaluation Agent & Semantic Ranking Engine
# ------------------------------------------------------------------


class EvaluationAgent:
    """Evaluates candidate answers against a 1-10 matrix and ranks candidates semantically."""

    def evaluate_candidate_answer(
        self,
        question_id: int,
        category: str,
        question: str,
        candidate_answer: str,
        expected_context: Optional[str] = None,
    ) -> QuestionGradingResult:
        """Grades a single candidate answer across accuracy, completeness, logic, and relevance."""
        ans_lower = candidate_answer.lower()
        word_count = len(candidate_answer.split())

        # Rule-based heuristic scoring engine (simulating LLM grading matrix)
        if word_count < 5:
            # Low quality / incomplete answer
            accuracy, completeness, logic, relevance = 3.0, 2.0, 2.0, 4.0
            explanation = "Answer is overly brief and lacks technical depth or logical rationale."
        else:
            # Check for domain keywords matching your candidate profile
            tech_keywords = [
                "rag",
                "embedding",
                "chunking",
                "prompt",
                "python",
                "ams",
                "jig",
                "cad",
                "optitex",
                "transformer",
            ]
            matched_keywords = [kw for kw in tech_keywords if kw in ans_lower]

            accuracy = min(10.0, 6.0 + len(matched_keywords) * 1.5)
            completeness = min(10.0, 5.0 + (word_count / 10.0))
            logic = 8.5 if "because" in ans_lower or "by" in ans_lower else 7.0
            relevance = 9.0 if len(matched_keywords) > 0 else 6.0

        # Weighted calculation for 1-10 composite score
        composite_score = round(
            (accuracy * 0.35)
            + (completeness * 0.25)
            + (logic * 0.20)
            + (relevance * 0.20),
            2,
        )

        explanation = (
            f"Candidate demonstrated clear practical knowledge matching {category.lower()} expectations. "
            f"Identified key operational concepts: {', '.join(matched_keywords) if 'matched_keywords' in locals() and matched_keywords else 'General overview'}."
        )

        return QuestionGradingResult(
            question_id=question_id,
            category=category,
            overall_score=composite_score,
            criteria_breakdown=EvaluationCriteria(
                technical_accuracy=round(accuracy, 1),
                completeness=round(completeness, 1),
                logical_rationale=round(logic, 1),
                relevance=round(relevance, 1),
            ),
            logical_explanation=explanation,
        )

    def calculate_candidate_rank(
        self,
        candidate_id: int,
        full_name: str,
        vector_similarity_score: float,  # From Qdrant match (0.0 to 1.0)
        evaluation_scores: List[float],  # List of 1.0 to 10.0 question grades
    ) -> CandidateRankResult:
        """Combines Phase 2 semantic vector similarity with Phase 4 evaluation grades for final ranking."""
        avg_eval_score = (
            sum(evaluation_scores) / len(evaluation_scores)
            if evaluation_scores
            else 0.0
        )

        # Scale assessment score to 0.0 - 1.0 scale to combine with vector similarity
        scaled_assessment = avg_eval_score / 10.0

        # Weighted Composite Matrix: 40% Vector Match + 60% Validated Q&A Evaluation
        final_rank = round(
            (vector_similarity_score * 0.40) + (scaled_assessment * 0.60), 4
        )

        summary = (
            f"Candidate {full_name} achieved a vector similarity match of {round(vector_similarity_score * 100, 1)}% "
            f"and an average Q&A assessment rating of {round(avg_eval_score, 2)}/10."
        )

        return CandidateRankResult(
            candidate_id=candidate_id,
            full_name=full_name,
            vector_similarity_score=vector_similarity_score,
            assessment_score=round(avg_eval_score, 2),
            final_weighted_rank_score=final_rank,
            ranking_summary=summary,
        )


# ------------------------------------------------------------------
# Test Phase 4 Workflow
# ------------------------------------------------------------------

eval_agent = EvaluationAgent()

# 1. Sample Question & Answer from Candidate
sample_question = "You mentioned experience with Generative AI and RAG architectures. Can you describe how you handled chunking or prompt engineering?"
sample_answer = "In my RAG workflow, I implemented semantic text chunking using Python and generated dense embeddings to improve context retrieval accuracy before feeding prompts into the LLM."

# 2. Grade Answer
grading_output = eval_agent.evaluate_candidate_answer(
    question_id=1,
    category="SKILL_VALIDATION",
    question=sample_question,
    candidate_answer=sample_answer,
)

print("--- Phase 4: Answer Evaluation Grading (1 to 10 Matrix) ---")
print(json.dumps(grading_output.model_dump(), indent=2))

# 3. Compute Final Semantic Rank
rank_output = eval_agent.calculate_candidate_rank(
    candidate_id=1,
    full_name="Deepan Raj K",
    vector_similarity_score=0.8412,  # Score from Qdrant in Phase 2
    evaluation_scores=[grading_output.overall_score, 8.8, 9.0],
)

print("\n--- Phase 4: Final Candidate Semantic Ranking ---")
print(json.dumps(rank_output.model_dump(), indent=2))

--- Phase 4: Answer Evaluation Grading (1 to 10 Matrix) ---
{
  "question_id": 1,
  "category": "SKILL_VALIDATION",
  "overall_score": 8.6,
  "criteria_breakdown": {
    "technical_accuracy": 10.0,
    "completeness": 7.6,
    "logical_rationale": 7.0,
    "relevance": 9.0
  },
  "logical_explanation": "Candidate demonstrated clear practical knowledge matching skill_validation expectations. Identified key operational concepts: rag, embedding, chunking, prompt, python."
}

--- Phase 4: Final Candidate Semantic Ranking ---
{
  "candidate_id": 1,
  "full_name": "Deepan Raj K",
  "vector_similarity_score": 0.8412,
  "assessment_score": 8.8,
  "final_weighted_rank_score": 0.8645,
  "ranking_summary": "Candidate Deepan Raj K achieved a vector similarity match of 84.1% and an average Q&A assessment rating of 8.8/10."
}


# 5.End-to-End API Routes & Orchestration

In [ ]:
import io
import json
import re
from datetime import datetime
from typing import Any, Dict, List, Optional

import fitz  # PyMuPDF
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from qdrant_client.http import models as qdrant_models
from sentence_transformers import SentenceTransformer
from sqlalchemy import (
    JSON,
    Column,
    DateTime,
    Float,
    ForeignKey,
    Integer,
    String,
    Text,
    create_engine,
)
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

# ==================================================================
# 1. Database & Vector DB Models
# ==================================================================

DATABASE_URL = "sqlite:///./phase5_orchestrated.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()


class CandidateProfile(Base):
    __tablename__ = "candidates"

    id = Column(Integer, primary_key=True, index=True)
    full_name = Column(String(150), nullable=False)
    email = Column(String(150), unique=True, index=True, nullable=False)
    phone = Column(String(50), nullable=True)
    location = Column(String(100), nullable=True)
    resume_json = Column(JSON, nullable=False)
    missing_data_report = Column(JSON, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

    assessments = relationship(
        "Assessment", back_populates="candidate", cascade="all, delete-orphan"
    )


class Assessment(Base):
    __tablename__ = "assessments"

    id = Column(Integer, primary_key=True, index=True)
    candidate_id = Column(Integer, ForeignKey("candidates.id"), nullable=False)
    target_role = Column(String(100), nullable=False)
    overall_score = Column(Float, nullable=True)
    status = Column(String(50), default="IN_PROGRESS")
    created_at = Column(DateTime, default=datetime.utcnow)

    candidate = relationship("CandidateProfile", back_populates="assessments")
    evaluations = relationship(
        "Evaluation", back_populates="assessment", cascade="all, delete-orphan"
    )


class Evaluation(Base):
    __tablename__ = "evaluations"

    id = Column(Integer, primary_key=True, index=True)
    assessment_id = Column(Integer, ForeignKey("assessments.id"), nullable=False)
    category = Column(String(50), nullable=False)
    question = Column(Text, nullable=False)
    candidate_answer = Column(Text, nullable=True)
    score = Column(Float, nullable=True)
    logical_explanation = Column(Text, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

    assessment = relationship("Assessment", back_populates="evaluations")


Base.metadata.create_all(bind=engine)


class VectorDBManager:

    def __init__(self, collection_name: str = "orchestrated_candidates"):
        self.collection_name = collection_name
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.vector_size = self.encoder.get_sentence_embedding_dimension()
        self.client = QdrantClient(":memory:")
        self._setup_collection()

    def _setup_collection(self):
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=qdrant_models.VectorParams(
                size=self.vector_size, distance=qdrant_models.Distance.COSINE
            ),
        )

    def upsert_candidate_vector(
        self, candidate_id: int, full_name: str, skills: List[str], work_summary: str
    ):
        text_repr = f"Candidate: {full_name}. Skills: {', '.join(skills)}. Experience: {work_summary}"
        vector = self.encoder.encode(text_repr).tolist()
        self.client.upsert(
            collection_name=self.collection_name,
            points=[
                qdrant_models.PointStruct(
                    id=candidate_id,
                    vector=vector,
                    payload={
                        "candidate_id": candidate_id,
                        "full_name": full_name,
                        "skills": skills,
                        "summary": work_summary,
                    },
                )
            ],
        )

    def search_candidates(
        self, role_requirements: str, limit: int = 5
    ) -> List[Dict[str, Any]]:
        query_vector = self.encoder.encode(role_requirements).tolist()
        results = self.client.query_points(
            collection_name=self.collection_name, query=query_vector, limit=limit
        ).points
        return [
            {
                "candidate_id": p.payload["candidate_id"],
                "full_name": p.payload["full_name"],
                "similarity_score": round(p.score, 4),
                "skills": p.payload["skills"],
            }
            for p in results
        ]


vector_db = VectorDBManager()

# ==================================================================
# 2. Pydantic Schemas
# ==================================================================


class AnswerSubmissionRequest(BaseModel):
    assessment_id: int
    category: str
    question: str
    candidate_answer: str


class EvaluationResponse(BaseModel):
    evaluation_id: int
    assessment_id: int
    score: float
    logical_explanation: str


# ==================================================================
# 3. FastAPI Application
# ==================================================================

app = FastAPI(title="Resume Parsing & AI Assessment Service", version="1.0.0")


# --- Endpoint 1: Parse PDF, Register Candidate (Using Form fields) ---
@app.post("/api/v1/parse-and-register")
async def parse_and_register(
    full_name: str = Form(...),
    email: str = Form(...),
    phone: Optional[str] = Form(None),
    file: UploadFile = File(...),
):
    if file.content_type != "application/pdf":
        raise HTTPException(status_code=400, detail="File must be a PDF document.")

    pdf_bytes = await file.read()
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    raw_text = "\n".join([page.get_text("text") for page in doc if page.get_text("text")])
    doc.close()

    if not raw_text.strip():
        raise HTTPException(status_code=422, detail="PDF content is empty or unreadable.")

    skills_taxonomy = [
        "Generative AI", "Python", "RAG", "LLM", "Deep Learning", "NLP", "Auto CAD", "Optitex 2D", "AMS Programming"
    ]
    extracted_skills = [
        s for s in skills_taxonomy if re.search(r"\b" + re.escape(s) + r"\b", raw_text, re.IGNORECASE)
    ]

    resume_json = {
        "education": [{"degree": "Degree Info", "institution": "Extracted University", "year": "2019"}],
        "work_experience": [{"title": "Engineer / CAD Specialist", "company": "Industry Experience", "duration": "2022 - 2026"}],
        "core_skills": extracted_skills,
    }

    missing_report = {
        "has_missing_data": len(extracted_skills) < 3,
        "missing_education_fields": [] if len(extracted_skills) >= 3 else ["Education CGPA or Institution detail missing"],
        "missing_skills": [] if extracted_skills else ["No skills matched taxonomy"],
    }

    db = SessionLocal()
    db.query(CandidateProfile).filter(CandidateProfile.email == email).delete()
    db.commit()

    candidate = CandidateProfile(
        full_name=full_name,
        email=email,
        phone=phone,
        location="Chennai, India",
        resume_json=resume_json,
        missing_data_report=missing_report,
    )
    db.add(candidate)
    db.commit()
    db.refresh(candidate)

    summary_text = f"Candidate {full_name} skilled in {', '.join(extracted_skills)}"
    vector_db.upsert_candidate_vector(
        candidate_id=candidate.id,
        full_name=full_name,
        skills=extracted_skills,
        work_summary=summary_text,
    )

    db.close()

    return {
        "status": "SUCCESS",
        "candidate_id": candidate.id,
        "parsed_skills": extracted_skills,
        "missing_data_report": missing_report,
    }


# --- Endpoint 2: Initiate Assessment ---
@app.post("/api/v1/assessment/initiate")
async def initiate_assessment(candidate_id: int, target_role: str):
    db = SessionLocal()
    candidate = db.query(CandidateProfile).filter(CandidateProfile.id == candidate_id).first()

    if not candidate:
        db.close()
        raise HTTPException(status_code=404, detail="Candidate not found.")

    assessment = Assessment(candidate_id=candidate.id, target_role=target_role, status="IN_PROGRESS")
    db.add(assessment)
    db.commit()
    db.refresh(assessment)

    questions = []
    skills = candidate.resume_json.get("core_skills", [])

    if candidate.missing_data_report.get("has_missing_data"):
        questions.append({
            "category": "GAP_FILLING",
            "question": "We observed some incomplete fields in your education history. Could you provide your exact degree title and completion score?"
        })

    if "Generative AI" in skills or "RAG" in skills:
        questions.append({
            "category": "SKILL_VALIDATION",
            "question": "How did you design your chunking and embedding pipelines when building RAG workflows in Python?"
        })

    questions.append({
        "category": "PROBLEM_SOLVING_SCENARIO",
        "question": f"Describe a significant technical bottleneck you resolved during your engineering experience for the {target_role} role."
    })

    db.close()

    return {
        "assessment_id": assessment.id,
        "candidate_id": candidate_id,
        "target_role": target_role,
        "generated_questions": questions,
    }


# --- Endpoint 3: Submit Answer & Grade ---
@app.post("/api/v1/assessment/submit-answer", response_model=EvaluationResponse)
async def submit_answer(payload: AnswerSubmissionRequest):
    db = SessionLocal()
    assessment = db.query(Assessment).filter(Assessment.id == payload.assessment_id).first()

    if not assessment:
        db.close()
        raise HTTPException(status_code=404, detail="Assessment session not found.")

    ans = payload.candidate_answer.lower()
    words = len(payload.candidate_answer.split())
    keywords = ["python", "rag", "embedding", "chunking", "jig", "cad", "ams"]
    matched = [k for k in keywords if k in ans]

    score = round(min(10.0, 5.0 + (len(matched) * 1.5) + (words / 15.0)), 2)
    explanation = f"Answer demonstrated good technical grasp. Key concepts identified: {', '.join(matched) if matched else 'General overview'}."

    eval_record = Evaluation(
        assessment_id=assessment.id,
        category=payload.category,
        question=payload.question,
        candidate_answer=payload.candidate_answer,
        score=score,
        logical_explanation=explanation,
    )
    db.add(eval_record)

    all_evals = db.query(Evaluation).filter(Evaluation.assessment_id == assessment.id).all()
    scores = [e.score for e in all_evals] + [score]
    assessment.overall_score = round(sum(scores) / len(scores), 2)

    db.commit()
    db.refresh(eval_record)
    db.close()

    return EvaluationResponse(
        evaluation_id=eval_record.id,
        assessment_id=payload.assessment_id,
        score=eval_record.score,
        logical_explanation=eval_record.logical_explanation,
    )


# --- Endpoint 4: Get Rankings ---
@app.get("/api/v1/candidates/rankings")
async def get_candidate_rankings(job_requirements: str):
    db = SessionLocal()
    vector_matches = vector_db.search_candidates(role_requirements=job_requirements, limit=5)

    rankings = []
    for match in vector_matches:
        cid = match["candidate_id"]
        latest_assessment = (
            db.query(Assessment)
            .filter(Assessment.candidate_id == cid)
            .order_by(Assessment.id.desc())
            .first()
        )
        assessment_score = (
            latest_assessment.overall_score
            if latest_assessment and latest_assessment.overall_score
            else 7.0
        )

        scaled_eval = assessment_score / 10.0
        final_rank_score = round((match["similarity_score"] * 0.40) + (scaled_eval * 0.60), 4)

        rankings.append({
            "candidate_id": cid,
            "full_name": match["full_name"],
            "vector_similarity_score": match["similarity_score"],
            "assessment_score_out_of_10": assessment_score,
            "final_weighted_rank_score": final_rank_score,
            "skills": match["skills"],
        })

    db.close()
    rankings.sort(key=lambda x: x["final_weighted_rank_score"], reverse=True)
    return {"query": job_requirements, "ranked_candidates": rankings}


# ==================================================================
# 4. Test Execution Block
# ==================================================================

client = TestClient(app)

sample_resume_text = """
DEEPAN RAJ K
Chennai, Tamil Nadu, India | +91 8939215805 | deepan98raj@gmail.com

PROFESSIONAL SUMMARY
Electronics and Communication Engineer skilled in Generative AI, Python, RAG, LLM, Optitex 2D, and AMS Programming.

CORE COMPETENCIES & TECHNICAL SKILLS
• AI & Machine Learning: Generative AI, RAG, LLM, Deep Learning
• Programming: Python, SQL
• CAD & Automation: Auto CAD, Optitex 2D, AMS Programming
"""

doc = fitz.open()
page = doc.new_page()
page.insert_text((40, 40), sample_resume_text, fontsize=9)
pdf_bytes = doc.write()
doc.close()

print("--- 1. Testing Parse & Register Endpoint ---")
reg_res = client.post(
    "/api/v1/parse-and-register",
    data={
        "full_name": "Deepan Raj K",
        "email": "deepan98raj@gmail.com",
        "phone": "+91 8939215805",
    },
    files={"file": ("Deepan_Resume.pdf", pdf_bytes, "application/pdf")},
)
print("Response:", reg_res.json())
candidate_id = reg_res.json()["candidate_id"]

print("\n--- 2. Testing Assessment Initiation Endpoint ---")
init_res = client.post(
    f"/api/v1/assessment/initiate?candidate_id={candidate_id}&target_role=AI%20Engineer"
)
print("Response:", json.dumps(init_res.json(), indent=2))
assessment_id = init_res.json()["assessment_id"]
questions = init_res.json()["generated_questions"]

print("\n--- 3. Testing Answer Submission & 1-10 Grading Endpoint ---")
sub_res = client.post(
    "/api/v1/assessment/submit-answer",
    json={
        "assessment_id": assessment_id,
        "category": questions[0]["category"],
        "question": questions[0]["question"],
        "candidate_answer": "I created semantic chunking pipelines in Python using SentenceTransformers to generate dense embeddings, which were stored in Qdrant for RAG context retrieval.",
    },
)
print("Response:", json.dumps(sub_res.json(), indent=2))

print("\n--- 4. Testing Candidate Semantic Rankings Endpoint ---")
rank_res = client.get(
    "/api/v1/candidates/rankings?job_requirements=Looking%20for%20an%20AI%20Engineer%20skilled%20in%20Python%2C%20Generative%20AI%2C%20and%20RAG%20pipelines"
)
print("Response:", json.dumps(rank_res.json(), indent=2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_4711/1737703662.py:93: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.vector_size = self.encoder.get_sentence_embedding_dimension()


--- 1. Testing Parse & Register Endpoint ---
Response: {'status': 'SUCCESS', 'candidate_id': 1, 'parsed_skills': ['Generative AI', 'Python', 'RAG', 'LLM', 'Deep Learning', 'Auto CAD', 'Optitex 2D', 'AMS Programming'], 'missing_data_report': {'has_missing_data': False, 'missing_education_fields': [], 'missing_skills': []}}

--- 2. Testing Assessment Initiation Endpoint ---
Response: {
  "assessment_id": 1,
  "candidate_id": 1,
  "target_role": "AI Engineer",
  "generated_questions": [
    {
      "category": "SKILL_VALIDATION",
      "question": "How did you design your chunking and embedding pipelines when building RAG workflows in Python?"
    },
    {
      "category": "PROBLEM_SOLVING_SCENARIO",
      "question": "Describe a significant technical bottleneck you resolved during your engineering experience for the AI Engineer role."
    }
  ]
}

--- 3. Testing Answer Submission & 1-10 Grading Endpoint ---
Response: {
  "evaluation_id": 1,
  "assessment_id": 1,
  "score": 10.0,
  "logi